# [EXPERIMENTAL] LoRA Empathy Fine-Tuning on ESConv & EmpatheticDialogues

**Notice:** This notebook is an experimental style adaptation module. In production, MANAS uses the server-side LLM pipeline with Pass A understanding.

### Pipeline Overview:
1. **Load Base Model in 4-bit NormalFloat (NF4) with BitsAndBytes**
2. **Prepare LoRA / PEFT Configuration (r=16, alpha=32, target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'])**
3. **Download and Format ESConv + EmpatheticDialogues into Alpaca/ShareGPT format**
4. **Train using SFTTrainer (HuggingFace TRL)**
5. **Evaluate Generation Quality vs. Gemini Baseline in the MANAS Evaluation Harness**

In [ ]:
# Step 1: Install PEFT, BitsAndBytes, TRL
!pip install -q transformers peft bitsandbytes datasets trl accelerate evaluate

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

MODEL_ID = 'meta-llama/Meta-Llama-3-8B-Instruct'  # or 'mistralai/Mistral-7B-Instruct-v0.3'

# 4-bit Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print(f'Loading {MODEL_ID} in 4-bit...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto'
)
model = prepare_model_for_kbit_training(model)

In [ ]:
# Step 2: Configure LoRA Adapter
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Step 3: Format ESConv data into Motivational Interviewing Instruction Format
def format_esconv_prompt(seeker_msg, strategy, supporter_reply):
    return (
        f'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n'
        f'You are an empathetic emotional companion adhering to ESConv. Strategy: {strategy}.<|eot_id|>'
        f'<|start_header_id|>user<|end_header_id|>\n{seeker_msg}<|eot_id|>'
        f'<|start_header_id|>assistant<|end_header_id|>\n{supporter_reply}<|eot_id|>'
    )

print('Formatted ESConv training format ready.')

In [ ]:
# Step 4: SFTTrainer configuration
training_args = TrainingArguments(
    output_dir='./results_esconv_lora',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim='paged_adamw_8bit',
    logging_steps=10,
    learning_rate=2e-4,
    fp16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine'
)

print('Ready to run SFTTrainer. Benchmark final model against Gemini using eval/run_eval.py')